In [0]:
### set the path and check if the files of yellow and reference files are present or not (paraquet and csv files)
catalog = "taxi"
base = f"/Volumes/{catalog}/raw/landing"
display(dbutils.fs.ls(f"{base}/reference/"))
display(dbutils.fs.ls(f"{base}/yellow/"))

In [0]:
### read the trip data
df = spark.read.parquet(f"{base}/yellow/")
df.printSchema()
print("rows: ",df.count())
display(df.limit(10))

In [0]:
### read the zone lookup
zones = spark.read.csv(f"{base}/reference/taxi_zone_lookup.csv",header=True,inferSchema=True)
zones.printSchema()
display(zones.limit(10))

In [0]:
### count nulls in each column
from pyspark.sql import functions as F
display(df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]))

In [0]:
### look for suspicious values
display(df.select(
    F.sum((F.col("fare_amount")<=0).cast("int")).alias("non_positive_fare"),
    F.sum((F.col("trip_distance")<=0).cast("int")).alias("non_positive_distance"),
    F.sum((F.col("tpep_dropoff_datetime")<=F.col("tpep_pickup_datetime")).cast("int")).alias("drop_off_before_pickup"),
    F.min("tpep_pickup_datetime").alias("min_pickup"),
    F.max("tpep_pickup_datetime").alias("max_pickup"),
))